# Love at First Agent: a LangGraph agent that catches its own data leakage

Can you predict whether two people will match at a speed-dating event **before they meet**?
This notebook trains XGBoost on the Columbia speed-dating experiment and gives a LangGraph
ReAct agent tools to profile the data, check features for leakage, train models and score a new couple.

The main lesson: most of the dataset's columns were recorded **during or after** each date
(ratings like `attr`, `like`, `prob`, and the partner's versions `attr_o`, `like_o`, ...).
A model that uses them scores about 0.86 ROC AUC, but it cannot be used before the date.
Using only what is known beforehand, the honest result is much lower, and that is reported below.

**Data:** download `Speed Dating Data.csv` from the Kaggle "Speed Dating Experiment" dataset
into `data/raw/` (the folder is git-ignored).
**LLM:** any OpenAI-compatible endpoint set in the repo's `.env` (LM Studio with Qwen works; see `.env.example`).

In [ ]:
# Cell 1 — Imports and settings
from dotenv import load_dotenv, find_dotenv
import os, re, json
load_dotenv(find_dotenv(usecwd=True))
os.environ.setdefault("OPENAI_API_KEY", "lm-studio")  # local servers accept any key

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split, GroupShuffleSplit, GroupKFold, cross_val_score
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
# Project folder, whether this runs from notebooks/ or the project root
PROJECT_DIR = Path.cwd().parent if Path.cwd().name in ("notebooks", "src") else Path.cwd()
OUTPUT_DIR = PROJECT_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

XGB_PARAMS = dict(n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8,
                  random_state=42, eval_metric='logloss', verbosity=0)
print("All imports successful")

In [ ]:
# Cell 2 — Load the dataset
RAW = PROJECT_DIR / 'data' / 'raw' / 'Speed Dating Data.csv'
if not RAW.exists():
    raise FileNotFoundError(
        f"Put 'Speed Dating Data.csv' from Kaggle's 'Speed Dating Experiment' dataset at {RAW}"
    )
df = pd.read_csv(RAW, encoding='ISO-8859-1')

print(f"Dataset shape: {df.shape}")
print(f"Match rate:    {df['match'].mean()*100:.1f}%  (imbalanced, so ROC AUC is the main metric)")

## Which columns are leakage?

Each row is one person's view of one date. Columns fall into three groups:

| Group | Examples | Known before the date? |
|---|---|---|
| IDs and bookkeeping | `iid`, `pid`, `wave`, `round`, `order` | not features |
| Outcome | `match`, `dec`, `dec_o` | no: this is the answer |
| Scorecard and follow-ups | `attr`, `like`, `prob`, `met`, their `_o` versions, `match_es`, `*_s`, `*_2`, `*_3` | no: filled in during or after the event |
| Sign-up survey | `age`, `age_o`, `int_corr`, `samerace`, stated preferences (`attr1_1`, `pf_o_att`, ...), interests | yes |

Only the last group can be used to predict a match before two people meet.

In [ ]:
# Cell 3 — Define leakage and build the feature sets
ID_COLS = ['iid', 'id', 'idg', 'condtn', 'wave', 'round', 'position', 'positin1', 'order', 'partner', 'pid']
OUTCOME = re.compile(r'^(match|dec|dec_o)$')
AFTER_DATE = re.compile(
    r'^(attr|sinc|intel|fun|amb|shar|like|prob|met)(_o)?$'   # scorecard ratings of this date
    r'|^match_es$|_s$'                                         # mid-event survey
    r'|_2$|_3$|^(you_call|them_cal|date_3|numdat_3|num_in_3|satis_2|length|numdat_2)$'  # follow-ups
)

def is_leakage(col: str) -> bool:
    return bool(OUTCOME.search(col) or AFTER_DATE.search(col))

y = df['match'].copy()
numeric = df.drop(columns=ID_COLS).select_dtypes(include=[np.number])
numeric = numeric.fillna(numeric.median())

naive_cols = [c for c in numeric.columns if not OUTCOME.search(c)]   # what the first version used
pre_cols   = [c for c in numeric.columns if not is_leakage(c)]       # known before the date

print(f"All numeric features:            {numeric.shape[1]}")
print(f"Without outcome columns (naive): {len(naive_cols)}")
print(f"Known before the date (honest):  {len(pre_cols)}")

# The same couple appears twice (once from each side). Group splits by couple so a
# couple never sits in both train and test.
couple = (df[['iid', 'pid']].min(axis=1).astype(str) + '-' + df[['iid', 'pid']].max(axis=1).astype(str))
train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(numeric, y, groups=couple))
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

In [ ]:
# Cell 4 — Naive model: outcome columns removed, but scorecard ratings kept
def rfe_top10(cols):
    rfe = RFE(RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1), n_features_to_select=10, step=5)
    rfe.fit(numeric.iloc[train_idx][cols], y_train)
    return [c for c, keep in zip(cols, rfe.support_) if keep]

def holdout_auc(cols):
    model = xgb.XGBClassifier(**XGB_PARAMS).fit(numeric.iloc[train_idx][cols], y_train)
    proba = model.predict_proba(numeric.iloc[test_idx][cols])[:, 1]
    return model, proba, roc_auc_score(y_test, proba)

naive_top10 = rfe_top10(naive_cols)
_, naive_proba, naive_auc = holdout_auc(naive_top10)
print("Naive top 10:", naive_top10)
print(f"Naive ROC AUC: {naive_auc:.4f}")
print("Leakage in the naive top 10:", [c for c in naive_top10 if is_leakage(c)])

In [ ]:
# Cell 5 — Honest model: only what is known before the date
honest_top10 = rfe_top10(pre_cols)
_, _, honest_top10_auc = holdout_auc(honest_top10)
xgb_model, honest_proba, honest_auc = holdout_auc(pre_cols)

cv_couple = cross_val_score(xgb.XGBClassifier(**XGB_PARAMS), numeric[pre_cols], y,
                            groups=couple, cv=GroupKFold(5), scoring='roc_auc')
cv_event = cross_val_score(xgb.XGBClassifier(**XGB_PARAMS), numeric[pre_cols], y,
                           groups=df['wave'], cv=GroupKFold(5), scoring='roc_auc')

print("Honest top 10 (RFE):", honest_top10)
print(f"Honest ROC AUC, top 10 features:        {honest_top10_auc:.4f}")
print(f"Honest ROC AUC, all {len(pre_cols)} pre-date features: {honest_auc:.4f}")
print(f"5-fold CV, grouped by couple:           {cv_couple.mean():.4f} ± {cv_couple.std():.4f}")
print(f"5-fold CV, grouped by event (new people): {cv_event.mean():.4f} ± {cv_event.std():.4f}")

In [ ]:
# Cell 6 — ROC curves: naive vs honest
fig = go.Figure()
for name, proba, auc_ in [("Naive, uses date ratings", naive_proba, naive_auc),
                          ("Honest, pre-date only", honest_proba, honest_auc)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=f'{name} (AUC={auc_:.3f})'))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', line=dict(color='gray', dash='dash'), name='Chance'))
fig.update_layout(title='ROC curves: what leakage does to a score', xaxis_title='False positive rate',
                  yaxis_title='True positive rate', template='plotly_dark', width=750, height=480)
fig.show()

## The agent

A LangGraph ReAct agent gets four tools. `check_leakage` uses the same rules as above, so it
flags scorecard ratings as well as the decision columns. `train_model` refuses leaky features
unless asked to show the naive score, and `predict_match` only takes pre-date inputs.

In [ ]:
# Cell 7 — Agent tools
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

# Reasoning models (for example Qwen in LM Studio) need extra tokens for their thinking.
def llm_budget(n): return n + int(os.getenv("LLM_REASONING_TOKENS", "0"))
def llm_text(text: str) -> str:
    return re.sub(r"<think>.*?</think>", "", text or "", flags=re.S).strip()

llm = ChatOpenAI(model=os.getenv("LLM_MODEL", "gpt-4o-mini"), temperature=0, max_tokens=llm_budget(1000))

@tool
def profile_data() -> str:
    """Profile the speed-dating dataset: size, match rate and the recommended metric."""
    return json.dumps({
        "rows": int(df.shape[0]), "columns": int(df.shape[1]),
        "match_rate": f"{y.mean()*100:.1f}%",
        "pre_date_features": len(pre_cols),
        "recommendation": "Use ROC AUC: only about 1 in 6 dates is a match.",
    })

@tool
def check_leakage(features: str) -> str:
    """Check a comma-separated list of column names for leakage (anything recorded during or after the date)."""
    names = [f.strip() for f in features.split(',') if f.strip()]
    flagged = [f for f in names if is_leakage(f)]
    return json.dumps({
        "flagged_leakage": flagged,
        "safe_features": [f for f in names if f not in flagged],
        "why": "Outcome columns and ratings filled in during or after the date are not known beforehand.",
    })

@tool
def train_model(features: str, allow_leakage: bool = False) -> str:
    """Train XGBoost on comma-separated features and return the held-out ROC AUC (split by couple).
    Leaky features are dropped unless allow_leakage is true."""
    names = [f.strip() for f in features.split(',') if f.strip() in numeric.columns]
    dropped = [] if allow_leakage else [f for f in names if is_leakage(f)]
    names = [f for f in names if f not in dropped]
    if not names:
        return json.dumps({"error": "No usable features left", "dropped_for_leakage": dropped})
    _, _, auc_ = holdout_auc(names)
    return json.dumps({"features_used": names, "dropped_for_leakage": dropped, "roc_auc": round(float(auc_), 4)})

@tool
def predict_match(age: float, partner_age: float, interest_correlation: float, same_race: bool) -> str:
    """Estimate the match probability for a new couple from pre-date information only.
    interest_correlation is between -1 and 1. Other inputs use typical values from the data."""
    row = numeric[pre_cols].median().to_frame().T
    row[['age', 'age_o', 'int_corr', 'samerace']] = [age, partner_age, interest_correlation, int(same_race)]
    prob = float(xgb_model.predict_proba(row[pre_cols])[0, 1])
    return json.dumps({
        "match_probability": f"{prob*100:.1f}%",
        "base_rate": f"{y.mean()*100:.1f}%",
        "note": f"Pre-date model, held-out ROC AUC {honest_auc:.2f}: a weak signal, not a forecast.",
    })

tools = [profile_data, check_leakage, train_model, predict_match]
print(f"Model: {llm.model_name} | tools: {[t.name for t in tools]}")

In [ ]:
# Cell 8 — Run the agent
agent = create_react_agent(llm, tools=tools)

queries = [
    "Profile the speed dating dataset and tell me if ROC AUC is the right metric.",
    "Check these features for leakage: attr_o, fun_o, dec, dec_o, like_o, shar_o, int_corr, age_o",
    "Train a model on attr_o, fun_o, like_o, int_corr, age, age_o. Then train the same list with allow_leakage "
    "set to true, and explain the difference.",
    "Estimate the match probability for a 28-year-old and a 26-year-old of the same race with interest correlation 0.4.",
]
agent_log = []
for q in queries:
    result = agent.invoke({"messages": [{"role": "user", "content": q}]})
    answer = llm_text(result["messages"][-1].content)
    calls = [tc["name"] for m in result["messages"] for tc in (getattr(m, "tool_calls", None) or [])]
    agent_log.append({"question": q, "tools_called": calls, "answer": answer})
    print(f"\nQ: {q}\nTools: {calls}\nAgent: {answer}\n" + "-" * 60)

In [ ]:
# Cell 9 — Export results
results = pd.DataFrame({
    'model': ['Naive: top 10 incl. date ratings', 'Honest: top 10 pre-date', 'Honest: all pre-date',
              'Honest: all pre-date, CV by couple', 'Honest: all pre-date, CV by event'],
    'roc_auc': [naive_auc, honest_top10_auc, honest_auc, cv_couple.mean(), cv_event.mean()],
    'std': [None, None, None, cv_couple.std(), cv_event.std()],
}).round(4)
results.to_csv(OUTPUT_DIR / 'model_results.csv', index=False)
json.dump({"naive_top10": naive_top10, "honest_top10": honest_top10, "pre_date_features": pre_cols},
          open(OUTPUT_DIR / 'features.json', 'w'), indent=2)
model_name = os.getenv("LLM_MODEL", "gpt-4o-mini")
json.dump({"model": model_name, "runs": agent_log}, open(OUTPUT_DIR / 'agent_run.json', 'w'), indent=2)
results